# Project Lab 1: Pipeline, Load Test and Anomaly Detection

Notebook for **Chapter 15** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to Run a Project in This Area

## 15.2 Project: End-to-End MLOps Pipeline (Train, Track, Deploy, Monitor)

In [ ]:
def train_one(name, model, X, y, params):
    n = int(len(X) * 0.8)                       # same split for every model
    t0 = time.time(); model.fit(X[:n], y[:n]); secs = time.time() - t0
    mae = mean_absolute_error(y[n:], model.predict(X[n:]))
    data_hash = hashlib.sha256(X.tobytes()).hexdigest()[:10]
    run = {"run_id": f"{name}-{int(time.time()*1000)%100000}", "model": name,
           "params": params, "metrics": {"mae_minutes": round(mae, 3),
           "train_seconds": round(secs, 3)}, "data_hash": data_hash}
    with LOG.open("a") as f: f.write(json.dumps(run) + "\n")
    return model, run

In [ ]:
def register_best(runs, models):
    best = min(runs, key=lambda r: r["metrics"]["mae_minutes"])
    reg = json.loads(REGISTRY.read_text()) if REGISTRY.exists() else {"versions": []}
    version = len(reg["versions"]) + 1
    reg["versions"].append({"version": version, "run_id": best["run_id"],
                            "stage": "production", "mae_minutes": best["metrics"]["mae_minutes"]})
    REGISTRY.write_text(json.dumps(reg, indent=2))
    return models[best["run_id"]], version

In [ ]:
body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
x = body["features"]
assert len(x) == 3 and all(isinstance(v, (int, float)) for v in x)
p = float(model.predict(np.array([x]))[0])
self._send(200, {"minutes": round(p, 2), "model_version": version})
# any exception -> errors += 1 and HTTP 422

In [ ]:
def test_registry_picks_lowest_mae():
    pipeline.REGISTRY = Path("runs/test_registry.json")     # a fresh file, not the real one
    pipeline.REGISTRY.unlink(missing_ok=True)
    runs = [{"run_id": "a", "metrics": {"mae_minutes": 3.0}},
            {"run_id": "b", "metrics": {"mae_minutes": 2.5}}]
    model, version = pipeline.register_best(runs, {"a": "model_a", "b": "model_b"})
    assert model == "model_b" and version == 1

## 15.3 Project: Load-Test a Model Endpoint

In [ ]:
def pct(sorted_vals, q):
    k = (len(sorted_vals) - 1) * q / 100
    lo, hi = int(k), min(int(k) + 1, len(sorted_vals) - 1)
    return sorted_vals[lo] + (sorted_vals[hi] - sorted_vals[lo]) * (k - lo)

In [ ]:
def user():
    conn = http.client.HTTPConnection("127.0.0.1", port)
    while time.time() < stop:
        t0 = time.perf_counter()
        conn.request("POST", "/predict", body, {"Content-Length": str(len(body))})
        r = conn.getresponse(); r.read()
        dt = (time.perf_counter() - t0) * 1000
        with lock:
            (lat.append(dt) if r.status == 200 else errors.append(1))

## 15.4 Project: Anomaly Detection on Service Metrics

In [ ]:
def rolling_z(x, window=60):
    score = np.zeros(len(x))
    for i in range(window, len(x)):
        w = x[i - window:i]
        c = np.median(w); s = 1.4826 * np.median(np.abs(w - c)) + 1e-9
        score[i] = (x[i] - c) / s
    return score

In [ ]:
def persist(flag, k):          # keep a flag only if it was on k minutes in a row
    out = np.zeros(len(flag), bool); run = 0
    for i, f in enumerate(flag):
        run = run + 1 if f else 0
        out[i] = run >= k
    return out

def to_alerts(flag, cooldown=30):   # one alert per burst; bursts within cooldown merge
    alerts, last = [], -10**9
    for i in np.flatnonzero(flag):
        if i - last > cooldown: alerts.append(i)
        last = i
    return alerts

## Chapter recap

## Practice set

**1. (easy)** Write one strong question with a number for a project that load-tests a model. **Answer:** For example: "At how many simultaneous users does p95 latency pass 50 ms, and what is the cost per 1,000 predictions there?" It has a measurable limit and a measurable cost.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch15.html